# Befektetési irányok elemzése
## Barna Péter Bálint
A notebook a cégek pontozását végzi el

In [ ]:
import re
import json
import requests
import helper_func # Saját készítésű függvények a feldolgozás segítésére

## Kulcsszavak
Itt definiáltam a kereséshez használt kulcsszavakat

In [ ]:
# Említésekhez használt kulcsszavak
TECH_KEYWORDS = {
    'CCUS': [
        'ccus', 'ccs', 'c_s', 'c_u', 'carbon capture', 'co2 capture',
        'carbon capture, utilization, and storage',
        'carbon capture, utilisation, and storage',
        'carbon sequestration', 'co2 sequestration', 'geological sequestration',
        'carbon storage', 'co2 storage',
        'carbon utilization', 'co2 utilization', 'carbon capture and utilization',
        'direct air capture', 'dac', 'd.a.c.',
        'co2 transport', 'co2 pipeline', 'carbon transport',
        'blue hydrogen',
        'post-combustion capture',
        'pre-combustion capture',
        'oxy-combustion',
        'solvent-based capture',
        'adsorption-based capture',
        'eor', 'enhanced oil recovery'
    ],
    'SMR': [
        'smr', 'small modular reactor',
        'advanced reactor', 'advanced nuclear',
        'micro-reactor', 'microreactor',
        'htgr', 'high-temperature gas reactor',
        'pbmr', 'pebble bed modular reactor',
        'molten salt reactor', 'msr',
        'liquid metal cooled reactor', 'lmr', 'lmfbr',
        'bwrx-300',   # Ismert SMR modellek
        'rolls-royce smr', 'rolls-royce small modular reactor',
        'nuscale',
        'terrestrial energy',
        'terrapower',
        'generation iv', 'gen iv'
    ],
    'Renewables': [
        'renewable energy', 'renewable power', 'clean energy',
        'ppa', 'power purchase agreement',
        'corporate ppa', 'cppa',
        'virtual ppa', 'vppa', 'sleeved ppa',
        'wind energy', 'wind power', 'wind farm',
        'onshore wind', 'offshore wind',
        'solar energy', 'solar power', 'solar farm', 'solar park',
        'photovoltaic', 'pv', 'solar pv',
        'agrivoltaics', 'agrovoltaics', 'solar grazing',
        'geothermal', 'geothermal energy',
        'battery storage', 'battery energy storage system', 'bess', 'grid-scale battery',
        're100',  # Céges kezdeményezés
        'green tariff', 'green energy tariff',
        'rooftop solar', 'on-site solar',
        'off-site renewable', 'offsite renewable',
        'renewable procurement', 'green energy procurement',
        'renewable energy certificates', 'recs', 'guarantees of origin', 'go-s',
        'hydropower', 'hydro-power',
        'bioenergy', 'biomass'
    ]
}

## Függvények
A keresés és a csoportosításhoz használt függvények találhatóak ebben a részben

In [ ]:
def google_search_api(company_name,API_KEY_, SEARCH_ENGINE_ID_, logging = False,):
    """ Az alábbi függvény kezeli egy adott cégre a 3 darab Google kereséset
    :param company_name: A vállalat neve
    :param API_KEY_: A kereséshez használható API kulcs
    :param SEARCH_ENGINE_ID_: A kereséshez használható search engine ID-je.
    """
    print(f"\n--- Fetching data for '{company_name}' via Google API ---")

    global TECH_KEYWORDS

    result = {
        "CCUS":[],
        "SMR":[],
        "Renewables":[]
    }
    for key, value in zip(TECH_KEYWORDS.keys(), TECH_KEYWORDS.values()):
        delimiter = ""
        query = f'{str.lower(company_name)} ' + "-blog" + " -forum " + delimiter.join(["\""+_+"\" OR " for _ in value])+"\""
        query = query[:-5]
        print(query) if logging else None
        url = f"https://www.googleapis.com/customsearch/v1?key={API_KEY_}&cx={SEARCH_ENGINE_ID_}&q={query}"
        search_results = []

        try:
            response = requests.get(url)
            response.raise_for_status()  # Raise an error for bad responses (4xx or 5xx)
            results = response.json()

            items = results.get('items', [])
            if not items:
                print(f"No results found for '{company_name}'.")
                result[key] = []

            for item in items:
                title = item.get('title','').lower()
                snippet = item.get('snippet', '').lower()

                flag = False

                # Az első 10 közül kiválasztom a megfelelőeket, amelyek tartalmazzák a céget és nevének variációit (generate name variants)
                compName = np.unique([i.lower() for i in helper_func.generate_name_variants(company_name,False)])
                print("\n**Following comp names were tested for**\n",compName) if logging else None
                print("\n**Type of comp names were tested for**\n",type(compName)) if logging else None
                for word in compName:
                    print("---------------------------------------------------------") if logging else None
                    print("------------ Checking |",word,"| in title |",title,"| ------") if logging else None
                    print("------------ Checking |",word,"| in snippet |",snippet,"| --") if logging else None
                    print(item.get('url','').lower()) if logging else None
                    print("---------------------------------------------------------") if logging else None
                    if word in title:
                        print("\n**TRUE**\n") if logging else None
                        flag = True
                        break
                    if word in snippet:
                        print("\n**TRUE**\n") if logging else None
                        flag = True
                        break
                    print("\n**FALSE**\n") if logging else None

                # Append snippet and title
                if snippet and flag:
                    search_results.append({'snippet': title + snippet.replace('\n', ' ')})

            print(f"Found {len(search_results)} relevant snippets.")
            result[key] = search_results

        except requests.exceptions.HTTPError as http_err:
            print(f"HTTP error occurred: {http_err}")
            print(f"Response content: {response.text}")
            if "API key not valid" in response.text:
                print("\n*** ERROR: Your API_KEY seems to be invalid. Check the Google Cloud Console. ***\n")
            elif "Invalid" in response.text:
                print("\n*** ERROR: Your CX_ID might be incorrect. Check your Programmable Search Engine settings. ***\n")
        except requests.exceptions.RequestException as err:
            print(f"An error occurred during the API request: {err}")
        except json.JSONDecodeError:
            print(f"Error decoding JSON response. Response text: {response.text}")

    return result

In [ ]:
def classify_companies(companies,API_KEY_, SEARCH_ENGINE_ID_, log=False):
    """ A kapott céglista feldolgozása és az eredmények táblázat létrehozása
    :param companies: 1D array, céglista, mely feldolgozásra kerül
    :param API_KEY_: A kereséshez használt API kulcs.
    :param SEARCH_ENGINE_ID_: A kereséshez használt custom search engine ID-je."""
    global TECH_KEYWORDS

    # Dataframe, mely tartalmazza a találatok számát (***_google), valamint a snippetek alapján végzett említéses számlálást (***_score)
    grandResult = pd.DataFrame(columns=["Company", "CCUS_google","SMR_google","Renewables_google","CCUS_score","SMR_score","Renewables_score"])

    for company in companies:
        googleSearchRes = google_search_api(company,API_KEY_,SEARCH_ENGINE_ID_,log)
        fultext = ""
        for item in googleSearchRes.values():
            for subitem in item:
                print(type(subitem["snippet"]))
                fultext += subitem["snippet"]

        # Run along the full text and classify it based on companies
        classif = {
            'CCUS':0,
            'SMR':0,
            'Renewables':0
        }
        for tech, keywords in TECH_KEYWORDS.items():
            mention_count = 0

            for keyword in keywords:
                mention_count += len(re.findall(r'\b' + re.escape(keyword) + r'\b', str.lower(fultext)))
            classif[tech] = mention_count

        vals = []
        for item in googleSearchRes.values():
            vals.append(len(item))


        #append result
        new_row_data = {
            "Company": company,
            "CCUS_google": vals[0],
            "SMR_google": vals[1],
            "Renewables_google": vals[2],
            "CCUS_score": classif["CCUS"],
            "SMR_score": classif["SMR"],
            "Renewables_score": classif["Renewables"]
        }
        grandResult.loc[len(grandResult)] = new_row_data

    return grandResult

## A keresések lefuttatása
Az alábbi részben lefuttatom a kereséseket a már meglévő cégekre

In [ ]:
import pandas as pd
import numpy as np

allData = pd.read_excel("Net0Data.xlsx")
nameArr = np.array(allData['name'])

In [ ]:
# A keresés megköveteli egy Google Custom Search engine, valamint egy hozzá tartozó érvényes API kulcs meglétét.
API_KEY = "YOUR_GOOGLE_API_KEY"
SEARCH_ENGINE_ID = "YOUR_CUSTOM_SEARCH_ENGINE_ID"

In [ ]:
CompanyData = classify_companies(nameArr[471:500],API_KEY,SEARCH_ENGINE_ID) #Mindig tömböt vagy listát küldj! 500:

In [ ]:
CompanyData.to_csv("adatgyujtes/CompanyScores471_500.txt") # Adatok kiíratása

In [ ]:
CompanyData

## A kiírt adatok összegyűjtése
Az alábbi részben az API kulcsok felhasználási limitje miatt külön kiíratott adatokat gyűjtöttem össze egy dataframe-be, majd excelbe.

In [ ]:
import glob

files = glob.glob('adatgyujtes/*.txt')
dataframes_list = [pd.read_csv(file, index_col=0) for file in files]
combined_df = pd.concat(dataframes_list, ignore_index=True)

In [ ]:
combined_df

In [ ]:
combined_df.rename(columns={'Company': 'name'}, inplace=True)
merged_df = pd.merge(allData, combined_df, on='name')

In [ ]:
merged_df.to_csv("Full_data_w_company_scoring_old.csv", index=False)

In [ ]:
import pandas as pd

In [ ]:
# Ellenőrzés
merged_df = pd.read_csv("Full_data_w_company_scoring_old.csv")
merged_df